<a href="https://colab.research.google.com/github/teguhsmlnna666/BigData26_B_2411532005_TeguhEsaMaulanna/blob/main/Praktikum5/BD_P05_2411532005_Teguh_Esa_Maulanna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Langkah Praktikum

H-1. Instalasi Library yang Dibutuhkan

Langkah ini digunakan untuk memasang library Python yang diperlukan dalam pelaksanaan praktikum, yaitu DuckDB, BeautifulSoup4, lxml, Requests, PyArrow, dan Pandas. Library tersebut mendukung proses pengolahan dan analisis data, pengambilan serta pemrosesan data dari halaman web, pembacaan file Parquet, dan manipulasi dataset. Parameter -q digunakan agar proses instalasi berjalan dengan keluaran yang lebih ringkas.

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

H-2. Pengambilan Dataset dan Pembuatan Portal Berita Lokal

Langkah ini digunakan untuk mengambil dataset judul berita berbahasa Indonesia dari GitHub, membaca data menggunakan Pandas, serta membangun portal berita lokal sebagai simulasi sumber data untuk proses web crawling. Portal dijalankan menggunakan HTTP server dengan fitur pembagian halaman sebanyak 100 judul berita per halaman. Selain itu, disediakan endpoint /robots.txt untuk memberikan aturan crawling, endpoint /arsip-privat/ sebagai simulasi halaman yang tidak boleh di-crawl, dan endpoint /berita untuk menampilkan judul berita beserta URL, kategori, dan tanggal publikasi.

In [2]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
"dataset-judul-berita-indonesia/master/detik_news_title.csv")

if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas:", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom:", df_sumber.shape)
print("Kolom:", list(df_sumber.columns))

import http.server, urllib.parse, html

HALAMAN_UKURAN = 100  # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass  # matikan log

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN : hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>'
                    )
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Ukuran berkas: 16.1 MB
Baris, kolom: (91017, 4)
Kolom: ['date', 'url', 'title', 'category']
Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


H-3. Pemeriksaan Robots.txt dan Implementasi Crawling Berita

Langkah ini digunakan untuk memeriksa aturan akses website melalui file robots.txt menggunakan modul urllib.robotparser. Program memeriksa apakah halaman /berita?hal=1 dan /arsip-privat/ diizinkan untuk diakses oleh crawler. Selanjutnya, dibuat fungsi parse_halaman() menggunakan BeautifulSoup untuk mengekstrak informasi berita berupa judul, URL, kategori, dan tanggal dari halaman HTML. Fungsi crawl_iter() kemudian digunakan untuk melakukan crawling secara bertahap dengan memeriksa izin akses, mengatur User-Agent, menangani percobaan ulang ketika terjadi kegagalan permintaan, serta memberikan jeda antarpermintaan.

In [3]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [4]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


H-4. Implementasi Parsing dan Crawling Sitemap Berita Detik.com

Langkah ini digunakan untuk pengambilan dan pengolahan data berita dari sitemap Detik.com. Proses dimulai dengan membuat fungsi parse_sitemap_news() untuk membaca data XML dan mengekstrak informasi berita, seperti judul, URL, kategori, ID, tanggal, dan kata kunci. Fungsi tersebut kemudian diuji menggunakan contoh XML untuk memastikan data dapat diproses dengan benar, baik dalam bentuk string maupun bytes. Selanjutnya, dilakukan crawling terhadap tiga sitemap Detik.com, yaitu kategori berita, teknologi, dan olahraga. Sebelum mengambil data, program memeriksa izin akses melalui robots.txt, menerapkan jeda 2 detik antarpermintaan, serta membatasi pengambilan maksimal 7 berita dari setiap sitemap dan 20 berita secara keseluruhan. Hasil crawling kemudian disimpan dalam format JSONL dan diperiksa menggunakan Pandas untuk mengetahui jumlah berita, kelengkapan data, duplikasi URL, serta distribusi kategori.

In [5]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else None, # news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": kunci.get_text(strip=True) if kunci else ""
        })
    return hasil

In [6]:
# Contoh XML: bentuk disalin dari sitemap berita asli (tiga entri hasil uji dosen, 29 Okt 2026).
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
<news:news>
<news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
<news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
<news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
<news:keywords><![CDATA[mahkamah konstitusi, mk, hukum, kepala daerah]]></news:keywords>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
<news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
</news:news>
</url>
<url>
<loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
<news:news>
<news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
<news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
</news:news>
</url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi, mk, hukum, kepala daerah
Sama bila masukan berupa bytes: True


In [7]:
import urllib.robotparser

UA_DETIK = "BigData/1.0 (praktikum pembelajaran; kontak: GANTI_DENGAN_EMAIL_ANDA)"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7       # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling

SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}

def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content  # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [8]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                                         judul kategori    tanggal
Usai Diperiksa KPK, Hercules Ungkap Asal-usul Uang Rp 800 Juta dari Summarecon     news 2026-11-11
                 BNPB Hentikan Water Bombing Karhutla di Kalsel Mulai Hari Ini     news 2026-11-11
 Kasus Penyadapan Prabowo: 8 Saksi Ajukan Perlindungan LPSK, 1 Orang Ajukan JC     news 2026-11-11


H-5. Implementasi Crawling Paralel dan Penyimpanan Data Mentah

Langkah ini digunakan untuk melakukan crawling dan penyimpanan data secara paralel menggunakan dua thread, yaitu source() sebagai pengambil data dan sink() sebagai penyimpan data. Keduanya berkomunikasi melalui antrean (queue.Queue) dengan kapasitas maksimal 2.000 data untuk mengatur aliran data. Data disimpan secara bertahap dalam beberapa file JSONL dengan maksimal 5.000 data per batch. Program juga mencatat jumlah data masuk dan keluar, kapasitas antrean maksimum, jumlah file, total ukuran penyimpanan, serta waktu proses crawling.

In [9]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)  # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu = time.perf_counter() - t0

print("Event masuk channel:", f"{statistik['masuk']:,}")
print("Event ditulis sink :", f"{statistik['keluar']:,}")
print("File batch         :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000)
print("Total ukuran mentah:", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total  :", round(waktu, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel: 91,017
Event ditulis sink : 91,017
File batch         : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah: 20.7 MB
Waktu crawl total  : 35.8 detik untuk 911 halaman


H-6. Implementasi Data Cleaning dan Penyimpanan Data Terkurasi

Langkah ini digunakan untuk proses pengolahan data mentah menjadi data terkurasi yang terdiri dari tiga bagian. Bagian pertama menggabungkan seluruh file JSONL menggunakan Pandas, kemudian memeriksa kualitas data berdasarkan kelengkapan judul, jumlah kata, kategori, validitas tanggal dan URL, serta duplikasi judul Selanjutnya melakukan pembersihan data dengan menghapus spasi berlebih, menghilangkan judul duplikat, menyaring judul berdasarkan jumlah kata, serta mengubah format tanggal dan mengekstrak ID artikel. Bagian ketiga menyimpan data bersih ke format Parquet dengan beberapa pilihan partisi, yaitu berdasarkan kategori, bulan, serta kombinasi kategori dan bulan. Serta, dilakukan pengujian partition pruning untuk melihat jumlah file yang perlu dibaca ketika mengambil data berdasarkan kategori tertentu.

In [10]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))], ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"  : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"     : jml_kata.between(3, 30),
        "kategori dikenal"    : d["kategori"].isin(KATEGORI_DIKENAL),
        "tanggal valid"       : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"   : d["url"].str.contains(r"/d-\d+", regex=True, na=False),
        "judul tidak duplikat": ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


In [11]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))  # kosong bila url rusak

print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [12]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

# BAGIAN KUNCI: bandingkan dua pilihan kolom partisi
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori finance


H-7. Analisis Data Menggunakan DuckDB dan Transformasi Dokumen JSON

Langkah ini digunakan untuk melakukan analisis data berita yang telah disimpan dalam format Parquet menggunakan DuckDB. Program membuat view bernama berita, kemudian menjalankan kueri SQL untuk menghitung jumlah berita berdasarkan kategori dan mencari jumlah judul yang mengandung kata "corona" berdasarkan bulan. Selain itu, program mengambil satu data berita dan mengubahnya menjadi dokumen JSON yang berisi ID, judul, kategori, tanggal, dan daftar kata dari judul berita.

In [13]:
import duckdb

con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")
print(con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df().to_string(index=False))
print()
print(con.sql("""
SELECT bulan, COUNT(*) AS judul_corona
FROM berita WHERE lower(judul) LIKE '%corona%'
GROUP BY bulan ORDER BY bulan""").df().to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


H-8. Simulasi MapReduce dan Combiner.

Langkah ini digunakan untuk menyimulasikan proses MapReduce menggunakan data transaksi dari tiga kasir. Proses terdiri dari tiga tahap, yaitu Map untuk memisahkan nama menu dan jumlah pesanan, Shuffle untuk mengelompokkan jumlah pesanan berdasarkan nama menu, serta Reduce untuk menjumlahkan seluruh pesanan pada setiap menu. Selain itu, diterapkan fungsi Combiner menggunakan collections.Counter untuk menjumlahkan pesanan di masing-masing kasir sebelum dikirim, sehingga jumlah pasangan data yang perlu diproses menjadi lebih sedikit.

In [14]:
kasir = [
    ["nasi goreng, 1", "es teh, 2", "mie ayam, 1", "es teh, 1"],       # kasir A
    ["nasi goreng, 2", "es teh, 1", "es teh, 3", "mie ayam, 1"],       # kasir B
    ["mie ayam, 2", "nasi goreng, 1", "es teh, 1", "nasi goreng, 1"],  # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


H-9. Implementasi Word Count Menggunakan MapReduce

Langkah ini digunakan untuk menghitung frekuensi kemunculan kata pada judul berita menggunakan konsep MapReduce. Dataset Parquet dibagi menjadi beberapa input split berdasarkan partisi kategori, kemudian setiap mapper membaca judul berita dan mengubahnya menjadi pasangan (kata, 1) melalui proses tokenisasi. Tahap Shuffle mengelompokkan pasangan berdasarkan kata, sedangkan Reduce menjumlahkan frekuensi setiap kata. Program juga menerapkan penyaringan stopword untuk menampilkan kata-kata yang lebih relevan serta melakukan pengecekan silang menggunakan collections.Counter untuk memastikan hasil perhitungan sesuai.

In [15]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")

top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


H-10. Perbandingan Kinerja Multiprocessing dan Combiner

Langkah ini digunakan untuk membandingkan kinerja proses MapReduce dengan dan tanpa Combiner menggunakan multiprocessing. Fungsi mapper_combiner() menerapkan proses penghitungan frekuensi kata pada setiap mapper sebelum hasilnya digabungkan. Selanjutnya, fungsi jalankan() digunakan untuk menjalankan proses secara sekuensial maupun paralel menggunakan 1 dan 2 proses. Setiap skenario diuji sebanyak tiga kali untuk memperoleh waktu eksekusi median dan jumlah pasangan data yang dikirim. Hasil akhirnya juga diperiksa untuk memastikan penggunaan Combiner tidak mengubah hasil perhitungan.

In [16]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")

for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")

    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.496              825,601
1 proses, dengan combiner         0.353               74,009
2 proses, tanpa combiner          0.671              825,601
2 proses, dengan combiner         0.390               74,009

Hasil dengan combiner sama persis: True


H-11. Analisis Partisi dan Distribusi Beban Reducer

Langkah ini digunakan untuk menganalisis pembagian beban data ke beberapa reducer menggunakan fungsi partitioner() berbasis CRC32. Jumlah reducer ditentukan sebanyak 4, kemudian setiap kata dipetakan ke reducer tertentu berdasarkan hasil hash. Program membandingkan distribusi beban reducer dengan dan tanpa Combiner, serta menguji ketimpangan beban pada 4, 16, dan 64 reducer. Selain itu, diperiksa lima kata dengan frekuensi tertinggi untuk mengetahui pengaruh kata yang sering muncul terhadap distribusi beban (data skew).

In [17]:
R = 4  # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", "  ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%)  R1=170,066 (20.6%)  R2=190,727 (23.1%)  R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%)  R1= 18,267 (24.7%)  R2= 18,530 (25.0%)  R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


H-12. Implementasi MapReduce Menggunakan Pipeline Linux

Langkah ini digunakan untuk mengimplementasikan proses MapReduce menggunakan pipeline Linux. Proses dimulai dengan menyiapkan data berita dalam format TSV, kemudian membuat program mapper.py untuk melakukan tokenisasi dan menghasilkan pasangan kata dengan nilai 1, serta reducer.py untuk menjumlahkan frekuensi setiap kata. Selanjutnya, data diproses melalui pipeline menggunakan perintah cat, sort, dan kedua program Python tersebut. Hasil penghitungan disimpan dalam file hasil_pipe.txt, kemudian ditampilkan lima kata dengan frekuensi tertinggi. Terakhir, dilakukan perbandingan dengan hasil MapReduce sebelumnya untuk memastikan kesesuaian hasil penghitungan.

In [18]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re

for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys

kunci_sekarang, total = None, 0

for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)

if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [19]:
!head -n 3 masukan.tsv
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5

finance	Kemnaker Awasi TKA di Meikarta
finance	BNI Digitalkan BNI Java Jazz 2020
finance	Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster

real	0m1.144s
user	0m1.230s
sys	0m0.062s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [20]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)

print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


H-13. Analisis Frekuensi Kata per Kategori dan Ekspor Data

Langkah ini digunakan untuk menghitung frekuensi kata berdasarkan kategori berita menggunakan konsep MapReduce. Fungsi mapper_kategori() membaca setiap partisi Parquet, melakukan tokenisasi judul, menyaring stopword, serta menghitung frekuensi kata dengan key gabungan kategori dan kata. Hasilnya digabungkan menggunakan Counter untuk menampilkan lima kata teratas pada setiap kategori. Selanjutnya, hasil penghitungan disimpan dalam format Parquet dan diekspor ke database SQLite sebagai simulasi proses Sqoop Export. Data kemudian diuji melalui kueri SQL untuk menampilkan tiga kata dengan frekuensi tertinggi pada kategori health.

In [21]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1
    return c

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

per_kategori = collections.defaultdict(list)
for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))

for kat in sorted(per_kategori):
    top5 = sorted(per_kategori[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for n, k in top5))

import sqlite3

SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)

print("Baris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)
Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Latihan

N-1. Stopword Domain

Langkah ini digunakan untuk menambahkan lima stopword khusus domain ke dalam daftar stopword yang sudah tersedia, yaitu rp, ri, the, foto, dan galeri. Daftar stopword disalin terlebih dahulu menggunakan STOPWORD.copy() agar daftar asli tidak berubah. Selanjutnya, fungsi mapper_kategori_latihan() membaca judul berita dari setiap file Parquet, mengambil kategori berdasarkan nama folder, melakukan tokenisasi, serta menghitung frekuensi kata yang tidak termasuk dalam daftar stopword tambahan. Hasil dari seluruh mapper kemudian digabungkan menggunakan collections.Counter dan dikelompokkan berdasarkan kategori berita untuk memperoleh lima kata dengan frekuensi tertinggi pada setiap kategori.

In [22]:
STOPWORD_LATIHAN = STOPWORD.copy()

# Tambahkan 5 stopword domain
tambahan_stopword = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_LATIHAN.update(tambahan_stopword)

def mapper_kategori_latihan(path):
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]

    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD_LATIHAN:
                c[(kategori, kata)] += 1
    return c

gabung_latihan = collections.Counter()
for c in map(mapper_kategori_latihan, splits):
    gabung_latihan.update(c)

per_kategori_latihan = collections.defaultdict(list)

for (kat, kata), n in gabung_latihan.items():
    per_kategori_latihan[kat].append((n, kata))

for kat in sorted(per_kategori_latihan):
    top5 = sorted(per_kategori_latihan[kat], reverse=True)[:5]
    print(f"{kat:<8}", ", ".join(
        f"{kata} ({jumlah:,})" for jumlah, kata in top5
    ))

print("\nStopword tambahan:", sorted(tambahan_stopword))

finance  corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)
food     makanan (699), makan (539), enak (358), bikin (343), rumah (311)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), film (489)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), indonesia (256), honda (255)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), dunia (299), turis (284), indonesia (269)

Stopword tambahan: ['foto', 'galeri', 'ri', 'rp', 'the']


N-2. Analisis Frekuensi Kata per Bulan

Langkah ini digunakan untuk menghitung frekuensi kemunculan kata pada judul berita berdasarkan bulan publikasi. Fungsi mapper_bulan() membaca kolom judul dan bulan dari setiap file Parquet, kemudian melakukan tokenisasi terhadap judul berita dan mengabaikan kata-kata yang termasuk dalam daftar STOPWORD. Setiap kata dihitung berdasarkan pasangan bulan dan kata menggunakan collections.Counter. Hasil dari seluruh mapper digabungkan, kemudian dikelompokkan berdasarkan bulan untuk memperoleh sepuluh kata dengan frekuensi tertinggi pada Januari dan Mei 2020.

In [23]:
def mapper_bulan(path):
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])

    c = collections.Counter()
    for _, baris in df_.iterrows():
        bulan = baris["bulan"]
        for kata in tokenisasi(baris["judul"]):
            if kata not in STOPWORD:
                c[(bulan, kata)] += 1

    return c

# Proses MapReduce
gabung_bulan = collections.Counter()

for c in map(mapper_bulan, splits):
    gabung_bulan.update(c)

# Kelompokkan berdasarkan bulan
per_bulan = collections.defaultdict(list)

for (bulan, kata), jumlah in gabung_bulan.items():
    per_bulan[bulan].append((jumlah, kata))

for bulan in ["2020-01", "2020-05"]:
    top10 = sorted(
        per_bulan[bulan], reverse=True
    )[:10]

    print(f"Top 10 kata bulan {bulan}:")
    for jumlah, kata in top10:
        print(f"{kata:<12} {jumlah:>7,}")

Top 10 kata bulan 2020-01:
banjir         1,220
rp               616
polisi           554
jokowi           516
tahun            493
baru             475
mobil            428
minta            427
kpk              415
indonesia        410
Top 10 kata bulan 2020-05:
corona         2,935
kasus            640
positif          628
pandemi          596
warga            565
lebaran          562
rp               558
psbb             522
covid            522
polisi           517


N-3. Perhitungan Bigram

Langkah ini digunakan untuk menghitung frekuensi kemunculan bigram, yaitu pasangan dua kata yang berurutan dalam judul berita. Fungsi mapper_bigram() membaca kolom judul dari setiap file Parquet, melakukan tokenisasi, kemudian membentuk pasangan dua kata yang berdekatan. Setiap pasangan menghasilkan nilai 1 sebagai penanda kemunculan. Selanjutnya, fungsi mapper_combiner_bigram() menggunakan collections.Counter untuk menghitung frekuensi bigram pada masing-masing mapper sebelum hasilnya digabungkan. Proses ini dilakukan untuk membandingkan jumlah data yang dihasilkan tanpa combiner dan dengan combiner.

In [24]:
def mapper_bigram(path):
    """Menghasilkan pasangan dua kata berurutan."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]

    hasil = []
    for teks in judul:
        kata = tokenisasi(teks)
        bigram = [
            f"{kata[i]} {kata[i+1]}"
            for i in range(len(kata) - 1)
        ]
        hasil.extend((b, 1) for b in bigram)

    return hasil

def mapper_combiner_bigram(path):
    """Menghitung bigram terlebih dahulu pada setiap mapper."""
    c = collections.Counter()

    for bigram, jumlah in mapper_bigram(path):
        c[bigram] += jumlah

    return c

# Tanpa combiner
hasil_bigram = [mapper_bigram(p) for p in splits]
total_bigram = sum(len(h) for h in hasil_bigram)

# Dengan combiner
hasil_bigram_combiner = [
    mapper_combiner_bigram(p) for p in splits
]
total_bigram_combiner = sum(
    len(c) for c in hasil_bigram_combiner
)

frekuensi_bigram = collections.Counter()
for c in hasil_bigram_combiner:
    frekuensi_bigram.update(c)

print("Jumlah bigram tanpa combiner :", f"{total_bigram:,}")
print("Jumlah bigram dengan combiner:", f"{total_bigram_combiner:,}")
print("Jumlah bigram unik           :", f"{len(frekuensi_bigram):,}")

print("\n10 bigram teratas:")
for bigram, jumlah in frekuensi_bigram.most_common(10):
    print(f"{bigram:<25} {jumlah:>7,}")

Jumlah bigram tanpa combiner : 734,783
Jumlah bigram dengan combiner: 411,217
Jumlah bigram unik           : 349,114

10 bigram teratas:
virus corona                2,915
corona di                   2,178
positif corona              1,804
di tengah                   1,258
pandemi corona              1,075
di rumah                    1,047
new normal                    917
di indonesia                  871
rp juta                       700
hari ini                      627


N-4. Perbandingan Jumlah Reducer

langkah ini digunakan untuk menganalisis pengaruh variasi jumlah reducer terhadap distribusi beban pemrosesan dalam MapReduce. Pengujian dilakukan dengan menggunakan tiga variasi jumlah reducer, yaitu 2, 8, dan 32. Fungsi beban() digunakan untuk membagi pasangan hasil mapper ke masing-masing reducer berdasarkan fungsi partitioner. Selanjutnya, program menghitung total data, beban reducer terberat, persentase beban terberat, serta persentase pembagian ideal agar distribusi data dapat dibandingkan.

In [25]:
for R in [2, 8, 32]:
    print(f"Jumlah Reducer (R) = {R}")

    beban_reducer = beban(hasil_map, R)

    total = sum(beban_reducer)
    beban_terberat = max(beban_reducer)
    bagian_terberat = beban_terberat / total * 100
    bagian_adil = 100 / R

    print("Beban setiap reducer:", beban_reducer)
    print(f"Reducer terberat: {bagian_terberat:.2f}%")
    print(f"Bagian ideal/adil: {bagian_adil:.2f}%")

Jumlah Reducer (R) = 2
Beban setiap reducer: [431563, 394038]
Reducer terberat: 52.27%
Bagian ideal/adil: 50.00%
Jumlah Reducer (R) = 8
Beban setiap reducer: [138978, 77144, 82045, 109178, 101858, 92922, 108682, 114794]
Reducer terberat: 16.83%
Bagian ideal/adil: 12.50%
Jumlah Reducer (R) = 32
Beban setiap reducer: [48073, 17153, 19911, 31279, 30981, 20621, 40913, 18083, 40656, 21146, 18779, 24102, 21112, 24563, 22946, 35697, 24718, 22521, 20868, 28404, 23478, 25782, 19506, 38028, 25531, 16324, 22487, 25393, 26287, 21956, 25317, 22986]
Reducer terberat: 5.82%
Bagian ideal/adil: 3.12%


N-5. Percobaan Crawling dengan Jeda Waktu

Langkah ini digunakan untuk melakukan pengujian crawling terhadap 50 halaman berita dengan jeda 0,05 detik pada setiap permintaan. Proses crawling dilakukan menggunakan fungsi crawl_iter() dengan rentang halaman 1 hingga 50. Fungsi time.perf_counter() digunakan untuk mencatat waktu sebelum dan sesudah crawling sehingga durasi proses dapat dihitung. Selain itu, program memperkirakan waktu yang diperlukan untuk mengambil seluruh 911 halaman berdasarkan asumsi satu detik untuk setiap halaman.

In [26]:
# Crawling 50 halaman dengan jeda 0,05 detik
t0 = time.perf_counter()

hasil_crawl_50 = list(
    crawl_iter(BASE, range(1, 51), delay=0.05)
)

waktu_50 = time.perf_counter() - t0

estimasi_911 = 911 * 1

print("Jumlah halaman       :", 50)
print("Jumlah data terkumpul:", len(hasil_crawl_50))
print("Waktu crawling 50    :", round(waktu_50, 2), "detik")
print("Estimasi jeda 911 halaman:",
      estimasi_911, "detik")
print("Estimasi dalam menit :",
      round(estimasi_911 / 60, 2), "menit")

Jumlah halaman       : 50
Jumlah data terkumpul: 5000
Waktu crawling 50    : 4.38 detik
Estimasi jeda 911 halaman: 911 detik
Estimasi dalam menit : 15.18 menit


N-6. Crawling Sitemap Kategori Lain

Langkah ini digunakan untuk mengambil dan memproses sitemap berita dari Detik.com menggunakan HTTP Session dan library BeautifulSoup. Sitemap diambil dari URL https://news.detik.com/berita/sitemap_news.xml melalui fungsi ambil_xml() yang telah dibuat sebelumnya. Data XML kemudian diperiksa untuk mengetahui ukuran file, struktur dokumen, dan jumlah URL berita yang tersedia. Selanjutnya, fungsi parse_sitemap_baru() dibuat untuk mengambil informasi berita dari elemen XML, yaitu judul, URL, tanggal publikasi, kategori, dan ID berita. Parser ini disesuaikan dengan struktur sitemap Detik.com yang menggunakan namespace news:.

In [29]:
import requests

URL_SITEMAP_BARU = "https://news.detik.com/berita/sitemap_news.xml"

sesi = requests.Session()
sesi.headers["User-Agent"] = UA_DETIK

xml_baru = ambil_xml(URL_SITEMAP_BARU, sesi)

print("Tipe data:", type(xml_baru))
print("Ukuran XML:", len(xml_baru), "bytes")

sup_baru = BeautifulSoup(xml_baru, "xml")

print("\nContoh struktur XML:")
print(sup_baru.prettify()[:3000])

print("\nJumlah URL:", len(sup_baru.find_all("url")))

Tipe data: <class 'bytes'>
Ukuran XML: 176285 bytes

Contoh struktur XML:
<?xml version="1.0" encoding="utf-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9" xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
 <url>
  <loc>
   https://news.detik.com/berita/d-8748693/bima-arya-ungkap-pentingnya-data-dalam-pengembangan-ai
  </loc>
  <news:news>
   <news:publication>
    <news:name>
     detikNews
    </news:name>
    <news:language>
     id
    </news:language>
   </news:publication>
   <news:publication_date>
    2026-11-11T17:38:00+07:00
   </news:publication_date>
   <news:title>
    Bima Arya Ungkap Pentingnya Data dalam Pengembangan AI
   </news:title>
   <news:keywords>
    bnr,kemendagri,data,kecerdasan artifisial
   </news:keywords>
  </news:news>
 </url>
 <url>
  <loc>
   https://news.detik.com/berita/d-8748679/usai-diperiksa-kpk-hercules-ungkap-asal-usul-uang-rp-800-juta-dari-summarecon
  </loc>
  <news:news>
   <news:publication>
    <news:name>
     d

In [30]:
def parse_sitemap_baru(xml_data):
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []

    for u in sup.find_all("url"):
        loc = u.find("loc")

        judul = u.find("news:title")
        waktu = u.find("news:publication_date")

        if judul is None:
            judul = u.find("title")

        if waktu is None:
            waktu = u.find("publication_date")

        if loc is None or judul is None:
            continue

        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)

        hasil.append({
            "judul": re.sub(
                r"\s+", " ", judul.get_text(strip=True)
            ),
            "url": url,
            "kategori": m.group(1) if m else None,
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,
            "kata_kunci": ""
        })

    return hasil

hasil_sitemap_baru = parse_sitemap_baru(xml_baru)[:20]

df_sitemap_baru = pd.DataFrame(hasil_sitemap_baru)

print("Jumlah berita berhasil diproses:", len(df_sitemap_baru))
display(df_sitemap_baru)

if not df_sitemap_baru.empty:
    print("\nPemeriksaan kualitas data:")

    hasil_aturan = aturan_kualitas(df_sitemap_baru)

    for nama, hasil in hasil_aturan.items():
        valid = hasil.sum()
        tidak_valid = len(hasil) - valid

        print(
            f"{nama:<25}: "
            f"{valid}/{len(hasil)} valid, "
            f"{tidak_valid} tidak valid"
        )
else:
    print("Tidak ditemukan berita yang sesuai dengan format parser.")

Jumlah berita berhasil diproses: 20


,judul,url,kategori,id,tanggal,kata_kunci
0,Bima Arya Ungkap Pentingnya Data dalam Pengemb...,https://news.detik.com/berita/d-8748693/bima-a...,news,8748693,2026-11-11,
1,"Usai Diperiksa KPK, Hercules Ungkap Asal-usul ...",https://news.detik.com/berita/d-8748679/usai-d...,news,8748679,2026-11-11,
2,BNPB Hentikan Water Bombing Karhutla di Kalsel...,https://news.detik.com/berita/d-8748678/bnpb-h...,news,8748678,2026-11-11,
3,Kasus Penyadapan Prabowo: 8 Saksi Ajukan Perli...,https://news.detik.com/berita/d-8748661/kasus-...,news,8748661,2026-11-11,
4,Dedie Puji 23 PNS Kota Bogor Purnabakti Usai P...,https://news.detik.com/berita/d-8748656/dedie-...,news,8748656,2026-11-11,
5,Pengacara Roy Suryo Tanya Ajudan: Pernah Lihat...,https://news.detik.com/berita/d-8748651/pengac...,news,8748651,2026-11-11,
6,"Stadion Maulana Yusuf di Serang Tak Terurus, P...",https://news.detik.com/berita/d-8748646/stadio...,news,8748646,2026-11-11,
7,UGM Pastikan Aparat Tak Masuk Kampus Saat Tang...,https://news.detik.com/berita/d-8748610/ugm-pa...,news,8748610,2026-11-11,
8,Lemhannas Dorong Penguatan Ketahanan Nasional ...,https://news.detik.com/berita/d-8748589/lemhan...,news,8748589,2026-11-11,
9,Wamenkop Dorong Koperasi Pesantren Jadi Pengge...,https://news.detik.com/berita/d-8748586/wamenk...,news,8748586,2026-11-11,



Pemeriksaan kualitas data:
judul tidak kosong       : 20/20 valid, 0 tidak valid
judul 3-30 kata          : 20/20 valid, 0 tidak valid
kategori dikenal         : 20/20 valid, 0 tidak valid
tanggal valid            : 20/20 valid, 0 tidak valid
url artikel valid        : 20/20 valid, 0 tidak valid
judul tidak duplikat     : 20/20 valid, 0 tidak valid


N-7. — Menghitung Kata Kunci Terbanyak

ini digunakan untuk menghitung frekuensi kemunculan kata kunci pada judul berita menggunakan pendekatan MapReduce. Pertama, data dari variabel sampel diubah menjadi DataFrame dengan mengambil kolom kategori dan kata_kunci, kemudian disimpan dalam format TSV sebagai input pemrosesan. Selanjutnya, program membuat dua file Python, yaitu mapper_kata_kunci.py dan reducer_kata_kunci.py. Mapper bertugas memisahkan kata kunci berdasarkan tanda koma, mengubahnya menjadi huruf kecil, dan menghasilkan pasangan data berupa kata kunci dengan nilai 1. Reducer kemudian menggabungkan kata kunci yang sama dan menjumlahkan seluruh frekuensi kemunculannya. Terakhir, hasil pengolahan dibaca kembali menggunakan Pandas dan diurutkan berdasarkan jumlah kemunculan dari yang terbesar.

In [31]:
df_kata_kunci = pd.DataFrame(sampel)

print(df_kata_kunci.columns.tolist())

df_kata_kunci = df_kata_kunci[
    ["kategori", "kata_kunci"]
].dropna(subset=["kata_kunci"])

df_kata_kunci.to_csv(
    "kata_kunci_input.tsv",
    sep="\t",
    index=False,
    header=False
)

print("Jumlah berita:", len(df_kata_kunci))
display(df_kata_kunci.head())

['judul', 'url', 'kategori', 'id', 'tanggal', 'kata_kunci']
Jumlah berita: 20


,kategori,kata_kunci
0,news,"kpk,hercules,pt summarecon,suap,hak guna bangunan"
1,news,"bnpb,karhutla"
2,news,"lpsk,prabowo subianto,penyadapan,justice colla..."
3,news,"bnr,pemkot bogor,purnabakti,dedie a. rachim"
4,news,"jokowi,kumis,hukum,roy suryo"


In [32]:
Path("mapper_kata_kunci.py").write_text(r'''import sys
import re

for baris in sys.stdin:
    bagian = baris.rstrip("\n").split("\t")

    if len(bagian) < 2:
        continue

    kategori, kata_kunci = bagian[0], bagian[1]

    for kata in kata_kunci.split(","):
        kata = kata.strip().lower()

        if kata:
            print(f"{kata}\t1")
''')

print("File mapper_kata_kunci.py berhasil dibuat.")

File mapper_kata_kunci.py berhasil dibuat.


In [33]:
Path("reducer_kata_kunci.py").write_text(r'''import sys

kata_sekarang = None
jumlah = 0

for baris in sys.stdin:
    kata, nilai = baris.rstrip("\n").split("\t")
    nilai = int(nilai)

    if kata == kata_sekarang:
        jumlah += nilai
    else:
        if kata_sekarang is not None:
            print(f"{kata_sekarang}\t{jumlah}")

        kata_sekarang = kata
        jumlah = nilai

if kata_sekarang is not None:
    print(f"{kata_sekarang}\t{jumlah}")
''')

print("File reducer_kata_kunci.py berhasil dibuat.")

File reducer_kata_kunci.py berhasil dibuat.


In [34]:
!cat kata_kunci_input.tsv | python mapper_kata_kunci.py | sort | python reducer_kata_kunci.py > hasil_kata_kunci.tsv

In [35]:
df_hasil_kata_kunci = pd.read_csv(
    "hasil_kata_kunci.tsv",
    sep="\t",
    names=["kata_kunci", "jumlah"]
)

df_hasil_kata_kunci = df_hasil_kata_kunci.sort_values(
    by="jumlah",
    ascending=False
).reset_index(drop=True)

print("10 Kata Kunci Terbanyak:")
display(df_hasil_kata_kunci.head(10))

10 Kata Kunci Terbanyak:


,kata_kunci,jumlah
0,jensen huang,3
1,ibl all indonesian,3
2,ibl all indonesian 2026,3
3,alice huang,2
4,telkom,2
5,bnr,2
6,ceo nvidia,2
7,danantara,1
8,basketball teladan cup,1
9,basket,1


In [36]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

DIR_SIMPAN = Path("/content/drive/MyDrive/BigData/Praktikum5")

folder = {
    ".jsonl": "lapisan_mentah",
    ".csv": "quality_cleaning",
    ".parquet": "lapisan_terkurasi",
    ".json": "query_modeling/dokumen",
    ".db": "siap_pakai/database",
    ".sqlite": "siap_pakai/database",
    ".sqlite3": "siap_pakai/database",
    ".tsv": "processing_mapreduce"
}

jumlah = 0

for file in Path("/content").rglob("*"):
    if file.is_file() and file.suffix.lower() in folder:
        if "drive" in file.parts or any(
            x in file.parts for x in [".git", "__pycache__", "sample_data"]
        ):
            continue

        tujuan = DIR_SIMPAN / folder[file.suffix.lower()] / file.name
        tujuan.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file, tujuan)
        jumlah += 1

print(f"Berhasil menyimpan {jumlah} file ke {DIR_SIMPAN}")

Mounted at /content/drive
Berhasil menyimpan 38 file ke /content/drive/MyDrive/BigData/Praktikum5
